In [81]:
include( "args.jl" )

using Optim

mutable struct Params
    # Input matrix dimensions.
    Nu::defInt      # Number of inputs per window.
    Mu::defInt      # Number of windows to consider.
    Ku::defInt      # Length of each window.
end

In [69]:
mutable struct State
    # Position variables.
    x::defFloat;  y::defFloat

    # Velocity variables.
    ẋ::defFloat;  ẏ::defFloat
end

function model(z::State, u::Vector{defFloat}; δt::defFloat=1e-2)
    # Unpack the velocity inputs.
    δx = z.ẋ;  δy = z.ẏ

    # Unpack the acceralation inputs.
    δẋ = u[1];  δẏ = u[2]

    # Return updated state variable.
    return State( z.x + δt*δx, z.y + δt*δy, z.ẋ + δt*δẋ, z.ẏ + δt*δẏ )
end

function window(params::Params, z0::State, ulist::Vector{defFloat}; δt::defFloat=1e-2)
    # Reshape the input vector.
    u = reshape( ulist, params.Mu, params.Nu )

    # Number of time-steps.
    Nt = params.Mu*params.Ku

    # Initialize stae vector.
    zlist = Vector{State}( undef, Nt + 1 )
    zlist[1] = z0

    # Expand control inputs over the prediction window.
    ûlist = [u[m,:] for m ∈ 1:params.Mu for _ ∈ 1:params.Ku]

    # Simulate the model for Nt time-steps.
    for t ∈ 1:Nt
        zlist[t+1] = model( zlist[t], ûlist[t]; δt=δt )
    end

    # Return the state vector.
    return zlist
end

window (generic function with 1 method)

In [ ]:
function cost(t0::defFloat, params::Params, zlist::Vector{State}; δt=1e-2)::defFloat
    # Number of time steps.
    Nt = params.Mu*params.Ku

    # Follow the arc of a sine wave.
    ẑlist = hcat( t0 .+ δt*(0:Nt-1), sin.( t0 .+ δt*(0:Nt-1) ) )

    # Compute the difference from the desired trajectory.
    return sum( (zlist[t].x - ẑlist[t,1])^2 + (zlist[t].y - ẑlist[t,2])^2 for t ∈ 1:Nt )/Nt
end

function createloss(params::Params, t0::defFloat, z0::State; δt=1e-2)::Function
    loss(u::Vector{defFloat})::defFloat =
        cost( t0, params, model( params, z0, u; δt=δt ) )
    return loss
end

cost (generic function with 1 method)

In [ ]:
# Initialize parameter variables.
params = Params( 2, 5, 100 )

# Initial condition.
z0 = State( 0, 0, 0, 0 )
u0 = [0.1, -0.1, 0.05, -0.1]

# Simulate the window.
zlist = window( params, z0, u0; δt=1e-2 )

# Compute the cost of the window.
cost( 0.0, params, zlist; δt=1e-2 )

1.7427067299714596

In [84]:
optimize( cost, u0, params )

MethodError: MethodError: no method matching optimize(::typeof(cost), ::Vector{Float64}, ::Params)
The function `optimize` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  optimize(::Any, ::Any, ::Any, !Matched::AbstractArray, !Matched::AbstractArray, !Matched::AbstractArray, !Matched::Optim.ConstrainedOptimizer, !Matched::Optim.Options)
   @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/ipnewton/interior.jl:230
  optimize(::Any, ::Any, ::Any, !Matched::AbstractArray, !Matched::AbstractArray, !Matched::AbstractArray, !Matched::Optim.ConstrainedOptimizer)
   @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/ipnewton/interior.jl:230
  optimize(::Any, ::Any, ::Any, !Matched::AbstractArray, !Matched::AbstractArray, !Matched::AbstractArray)
   @ Optim ~/.julia/packages/Optim/7krni/src/multivariate/solvers/constrained/ipnewton/interior.jl:230
  ...
